# ReSnap — Retrain v4b di Kaggle (regularisasi lebih kuat)

**Kenapa ada v4b:** v4 (YOLOv8s + data Open Images) jauh lebih baik dari v3 di val (mAP50-95 0.426 → 0.480) dan di holdout (AP50 kentang 0.01 → 0.56, udang 0.14 → 0.39), tetapi gap mAP50-95 train-val-nya 0.35, jauh di atas v3 (0.22). Kurvanya menunjukkan loss train terus turun setelah sekitar epoch 50, sementara loss val dan mAP val mendatar: model mulai menghafal.

**Perubahan v4b terhadap v4** (data, val, dan holdout sama):
| | v4 | v4b | Alasan |
|---|---|---|---|
| Duplikat oversampling | 1 salinan | **dihapus** | Duplikat mendorong hafalan foto yang sama |
| mosaic | 0.5 | **1.0** | Variasi komposisi lebih besar |
| mixup | 0.0 | **0.1** | Regularisasi ringan |
| scale | 0.4 | **0.5** | Variasi ukuran objek |
| weight_decay | 0.0005 | **0.001** | Bobot lebih kecil = lebih umum |
| epoch maks / patience | 100 / 20 | **70 / 15** | Val v4 sudah mendatar sekitar epoch 55 |

**Persiapan:** Add Input dataset `resnap-v4` **dan** output notebook run v4 yang pertama (Add Input > Your Work > notebook `notebookd95b407e6e`), karena bobot v4 ikut dinilai ulang. Accelerator GPU T4, Internet On, lalu Save & Run All (Commit). Perkiraan 3–4 jam.

**Pemilihan model:** v4b dipakai bila lolos 4 syarat yang sama. Bila tidak, dipilih model dengan gap terkecil di antara v4 dan v4b yang val-nya tidak lebih buruk dari v3, dan statusnya ditulis "lolos sebagian". Test set hanya dievaluasi untuk model terpilih.

## 1. Paket, input, dan konfigurasi

In [ ]:
%pip -q install ultralytics==8.3.228

In [ ]:
from pathlib import Path
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working')
TMP = Path('/tmp/resnap')          # data besar di sini supaya tidak ikut Output
DEVICE = '0'

In [ ]:
import json, random, shutil, sys, zipfile
from collections import Counter

import torch
assert torch.cuda.is_available(), 'GPU tidak aktif. Settings > Accelerator > GPU, lalu jalankan ulang.'
print('GPU:', torch.cuda.get_device_name(0))

def find_first(pred, what):
    hits = sorted((p for p in INPUT.rglob('*') if pred(p)), key=lambda p: len(p.parts))
    if not hits:
        raise FileNotFoundError(f'{what} tidak ditemukan di {INPUT}. Cek Add Input.')
    return hits[0]

def materialize(zip_name, is_root, dest, what):
    # Pakai folder hasil ekstrak Kaggle bila ada, kalau tidak ekstrak ZIP-nya ke dest.
    if dest.exists() and is_root(dest):
        return dest
    try:
        src = find_first(lambda p: p.is_dir() and is_root(p), what)
        shutil.copytree(src, dest, dirs_exist_ok=True)
    except FileNotFoundError:
        z = find_first(lambda p: p.name == zip_name, what)
        with zipfile.ZipFile(z) as f:
            f.extractall(dest)
        inner = [p for p in dest.rglob('*') if p.is_dir() and is_root(p)]
        if not is_root(dest) and inner:  # ZIP berisi satu folder induk
            return sorted(inner, key=lambda p: len(p.parts))[0]
    assert is_root(dest), f'Struktur {what} tidak dikenali'
    return dest

TMP.mkdir(parents=True, exist_ok=True)
REPO = materialize('resnap.zip', lambda p: (p / 'scripts/data_v4.py').is_file(), TMP / 'repo', 'Repo resnap')
base = materialize('ReSnap_Kandidat_Tambah_Foto_Udang.zip',
                   lambda p: (p / 'test/images').is_dir() and (p / 'reports/class_catalog.json').is_file(),
                   TMP / 'base', 'Dataset kandidat v3')
data = materialize('dataset_v4.zip', lambda p: (p / 'holdout_oi/images').is_dir(), TMP / 'data', 'Dataset v4')

sys.path.insert(0, str(REPO / 'scripts'))
import data_v4 as dv
from diagnose_fit import analyze_curves, load_results, plot as plot_curves, verdict
from evaluate_export import evaluate
from ultralytics import YOLO

V3_WEIGHTS = REPO / 'models/best.pt'
V3_SHA = 'ac24d434cc97d12f3fd3ed204d2fefda8cccfc322422a8c55672190a0615da98'
assert dv.sha256_file(V3_WEIGHTS) == V3_SHA, 'models/best.pt di repo bukan v3'

EXPECTED = {'base': {'train': 4500, 'val': 1069, 'test': 654},
            'data': {'train': 5318, 'val': 1069, 'holdout_oi': 78}}
found = {'base': {s: len(list((base / s / 'images').iterdir())) for s in EXPECTED['base']},
         'data': {s: len(list((data / s / 'images').iterdir())) for s in EXPECTED['data']}}
print(found)
assert found == EXPECTED, 'Jumlah foto tidak cocok dengan dataset yang dibuat di Colab'

RUN_PROJECT = OUTPUT / 'training'
RUN_NAME = 'resnap_yolov8s_30kelas_v4b_reg'
RUN_DIR = RUN_PROJECT / RUN_NAME
OUT = OUTPUT / 'evaluasi_v4b'
OUT.mkdir(parents=True, exist_ok=True)
WORK = TMP
OVERSAMPLE_COPIES = 0
OI_TRAIN_REQUEST = {'Potato': 700, 'Shrimp': 700, 'Mushroom': 350}

# Hapus duplikat oversampling (dibuat untuk v4)
dups = [p for p in (data / 'train/images').iterdir() if '_dup' in p.stem]
for p in dups:
    p.unlink()
    (data / 'train/labels' / f'{p.stem}.txt').unlink(missing_ok=True)
print('Duplikat dihapus:', len(dups))
assert not any('_dup' in p.stem for p in (data / 'train/labels').iterdir())

# Bobot v4 dari output run pertama (dinilai ulang dengan subset yang sama)
V4_SHA = '1ee148a52fb8cef0a95bf207b3708ddcd4db06134d973aa9ae2db721ece109a5'
V4_PREV = find_first(lambda p: p.suffix == '.pt' and p.is_file() and dv.sha256_file(p) == V4_SHA,
                     'Bobot v4 (Add Input > Your Work > notebook run v4)')
print('Bobot v4:', V4_PREV)

yaml_train = dv.write_yaml(data / 'data.yaml', data, 'train/images', 'val/images')
yaml_holdout = dv.write_yaml(data / 'holdout.yaml', data, 'holdout_oi/images', 'holdout_oi/images')
stats = {'train_v3': dv.count_split(base / 'train'), 'train_v4': dv.count_split(data / 'train'),
         'val': dv.count_split(data / 'val'), 'holdout_oi': dv.count_split(data / 'holdout_oi')}
dv.save_json(OUT / 'jumlah_data.json', stats)
print('Siap. Kentang train:', stats['train_v4']['per_class']['potato'],
      '| udang train:', stats['train_v4']['per_class']['shrimp'])

## 2. Training YOLOv8s dengan regularisasi lebih kuat

In [ ]:
RESUME = False   # di Kaggle, run selalu dimulai baru
MAX_EPOCHS = 70
RECIPE = dict(
    epochs=MAX_EPOCHS, patience=15, batch=16, imgsz=640, save=True, save_period=5, cache=False,
    workers=2, pretrained=True, optimizer='AdamW', lr0=0.0003, lrf=0.01, momentum=0.937,
    weight_decay=0.001, warmup_epochs=3.0, seed=42, deterministic=True, cos_lr=False,
    close_mosaic=10, amp=True, plots=True, val=True,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.35, degrees=10.0, translate=0.1, scale=0.5, shear=0.0,
    perspective=0.0, flipud=0.0, fliplr=0.5, mosaic=1.0, mixup=0.1, copy_paste=0.0,
)
if RESUME:
    YOLO(str(RUN_DIR / 'weights/last.pt')).train(resume=True, device=DEVICE)
else:
    assert not RUN_DIR.exists(), f'{RUN_DIR} sudah ada. Ganti RUN_NAME atau set RESUME=True.'
    RUN_DIR.mkdir(parents=True)
    dv.save_json(RUN_DIR / 'experiment.json', {'base_model': 'yolov8s.pt', 'settings': RECIPE,
                                               'oversample_copies': OVERSAMPLE_COPIES,
                                               'oi_train_request': OI_TRAIN_REQUEST,
                                               'train_images': stats['train_v4']['images']})
    YOLO('yolov8s.pt').train(data=str(yaml_train), project=str(RUN_PROJECT), name=RUN_NAME,
                             exist_ok=True, device=DEVICE, **RECIPE)
V4B_WEIGHTS = RUN_DIR / 'weights/best.pt'
dv.save_json(RUN_DIR / 'training_complete.json', {'weights_sha256': dv.sha256_file(V4B_WEIGHTS)})
print('Selesai:', V4B_WEIGHTS)

## 3. Diagnosa overfit / underfit (v3, v4, v4b dengan subset yang sama)

In [ ]:
V4B_WEIGHTS = RUN_DIR / 'weights/best.pt'
rng = random.Random(42)

def subset_yaml(src_dir, n, name):
    imgs = sorted(p for p in (src_dir / 'images').iterdir() if '_dup' not in p.stem)
    pick = rng.sample(imgs, min(n, len(imgs)))
    dst = WORK / name
    if dst.exists():
        shutil.rmtree(dst)
    (dst / 'images').mkdir(parents=True); (dst / 'labels').mkdir()
    for p in pick:
        shutil.copy2(p, dst / 'images' / p.name)
        shutil.copy2(src_dir / 'labels' / f'{p.stem}.txt', dst / 'labels' / f'{p.stem}.txt')
    return dv.write_yaml(dst / 'data.yaml', dst, 'images', 'images')

yaml_tr_v4 = subset_yaml(data / 'train', 600, 'train_subset_v4')   # dipakai v4 dan v4b
yaml_tr_v3 = subset_yaml(base / 'train', 600, 'train_subset_v3')

MODELS = {'v3': V3_WEIGHTS, 'v4': V4_PREV, 'v4b': V4B_WEIGHTS}
E = {}
for name, w in MODELS.items():
    E[f'{name}_val'] = evaluate(str(w), yaml_train, 'val', OUT, f'{name}_val', DEVICE)
    E[f'{name}_train'] = evaluate(str(w), yaml_tr_v3 if name == 'v3' else yaml_tr_v4, 'val', OUT,
                                  f'{name}_train_subset', DEVICE)
    E[f'{name}_holdout'] = evaluate(str(w), yaml_holdout, 'val', OUT, f'{name}_holdout', DEVICE)
gap = {n: E[f'{n}_train']['mAP50_95'] - E[f'{n}_val']['mAP50_95'] for n in MODELS}

rows = load_results(RUN_DIR / 'results.csv')
curves = analyze_curves(rows, MAX_EPOCHS)
fit = verdict(curves, E['v4b_train']['mAP50_95'], E['v4b_val']['mAP50_95'], gap['v3'])
plot_curves(rows, OUT / 'kurva_v4b.png', curves['best_epoch'], 'Kurva training v4b (regularisasi kuat)')
dv.save_json(OUT / 'diagnosa_fit.json', {'kurva': curves, 'gap': gap, **fit, 'evaluasi': E})
print(json.dumps({'kurva': curves, 'gap': {k: round(v, 4) for k, v in gap.items()}, **fit},
                 indent=2, ensure_ascii=False))
from IPython.display import Image, display
display(Image(str(OUT / 'kurva_v4b.png'), width=900))

## 4. Bandingkan v3, v4, v4b

In [ ]:
import pandas as pd
names = list(MODELS)
overall = pd.DataFrame({n: E[f'{n}_val'] for n in names}).loc[
    ['precision', 'recall', 'f1', 'mAP50', 'mAP75', 'mAP50_95']]
overall.loc['gap_train_val'] = [gap[n] for n in names]
overall['delta_pp'] = 100 * (overall['v4b'] - overall['v3'])
display(overall.style.format(precision=3).set_caption('Val (gap = mAP50-95 train-subset dikurangi val)'))

def pc(name):
    return pd.read_csv(OUT / name / 'per_class.csv').set_index('class_name')

pv = {n: pc(f'{n}_val') for n in names}
cmp = pd.DataFrame({'instances': pv['v4b']['instances'], **{f'AP50_{n}': pv[n]['AP50'] for n in names}})
cmp['delta_pp'] = 100 * (cmp['AP50_v4b'] - cmp['AP50_v3'])
display(cmp.sort_values('delta_pp').style.format(precision=3).set_caption('AP50 per kelas di val'))

ph = {n: pc(f'{n}_holdout') for n in names}
focus = [c for c in ('potato', 'shrimp', 'mushroom') if c in ph['v4b'].index]
hold = pd.DataFrame({f'AP50_{n}': ph[n].reindex(focus)['AP50'].fillna(0) for n in names})
hold['delta_AP50_pp'] = 100 * (hold['AP50_v4b'] - hold['AP50_v3'])
display(hold.style.format(precision=3).set_caption('Holdout Open Images'))
cmp.to_csv(OUT / 'banding_per_kelas_val.csv'); hold.to_csv(OUT / 'banding_holdout.csv')
overall.to_csv(OUT / 'banding_keseluruhan.csv')

## 5. Keputusan

v4b diterima bila lolos 4 syarat yang sama seperti v4 (dibandingkan dengan v3). Bila tidak, dipilih model dengan gap terkecil di antara v4 dan v4b yang mAP50-95 val-nya tidak lebih buruk dari v3, dengan status *lolos sebagian*.

In [ ]:
checks = {
    'fit_seimbang': fit['vonis'] == 'seimbang',
    'val_map_tidak_turun': overall.loc['mAP50_95', 'delta_pp'] >= -0.5,
    'kentang_naik': 'potato' in hold.index and hold.loc['potato', 'delta_AP50_pp'] >= 5,
    'udang_naik': 'shrimp' in hold.index and hold.loc['shrimp', 'delta_AP50_pp'] >= 5,
    'tidak_ada_kelas_anjlok': not ((cmp['instances'] >= 20) & (cmp['delta_pp'] < -10)).any(),
}
ACCEPT = all(checks.values())
for k, v in checks.items():
    print(('LULUS ' if v else 'GAGAL ') + k)

if ACCEPT:
    CHOSEN, status = 'v4b', 'lolos penuh'
else:
    ok = [n for n in ('v4b', 'v4') if E[f'{n}_val']['mAP50_95'] >= E['v3_val']['mAP50_95'] - 0.005]
    CHOSEN = min(ok, key=gap.get) if ok else 'v3'
    status = 'lolos sebagian (gap masih di atas batas)' if ok else 'v4/v4b tidak lebih baik dari v3'
print(f"\nGap train-val: v3 {gap['v3']:.3f} | v4 {gap['v4']:.3f} | v4b {gap['v4b']:.3f}")
print(f'Model terpilih: {CHOSEN} ({status})')
dv.save_json(OUT / 'keputusan.json', {'accept_v4b': ACCEPT, 'checks': {k: bool(v) for k, v in checks.items()},
                                      'chosen': CHOSEN, 'status': status,
                                      'gap': {k: round(v, 4) for k, v in gap.items()}})

## 6. Evaluasi test final (sekali) untuk model terpilih

In [ ]:
import subprocess
chosen_weights = MODELS[CHOSEN]
final_dir = OUTPUT / f'evaluasi_final_{CHOSEN}'
weights_copy = OUTPUT / f'best_{CHOSEN}.pt'
shutil.copy2(chosen_weights, weights_copy)

test_yaml = dv.write_yaml(TMP / 'test.yaml', base, 'train/images', 'val/images', {'test': 'test/images'})
for split in ('test', 'val'):
    cmd = [sys.executable, str(REPO / 'scripts/evaluate_export.py'), '--weights', str(weights_copy),
           '--data', str(test_yaml), '--split', split, '--out', str(final_dir / split),
           '--device', DEVICE, '--ablation'] + (['--onnx'] if split == 'test' else [])
    subprocess.run(cmd, check=True, cwd=REPO)

final = {s: json.loads((final_dir / s / 'ringkasan.json').read_text()) for s in ('test', 'val')}
cols = ['images', 'precision', 'recall', 'f1', 'mAP50', 'mAP75', 'mAP50_95']
display(pd.DataFrame({'test (mentah)': final['test']['raw'], 'test (denoise+CLAHE)': final['test']['enhanced'],
                      'val (mentah)': final['val']['raw'], 'val (denoise+CLAHE)': final['val']['enhanced']})
        .loc[cols].style.format(precision=3).set_caption(f'Model {CHOSEN}'))
print('Selisih ablasi test (poin):', final['test']['delta_pp'])
print('Selisih ablasi val  (poin):', final['val']['delta_pp'])
per_cls = pd.read_csv(final_dir / 'test/test_raw/per_class.csv').sort_values('AP50')
display(per_cls.head(8).style.format(precision=3).set_caption('8 kelas terlemah di test'))
from IPython.display import Image
for f in ('confusion_matrix_normalized.png', 'BoxPR_curve.png'):
    display(Image(str(final_dir / 'test/test_raw' / f), width=700))

## 7. Kemas hasil

Unduh `hasil_v4b.zip` dari tab **Output**.

In [ ]:
pack = OUTPUT / 'hasil_v4b.zip'
with zipfile.ZipFile(pack, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT.rglob('*'):
        if p.is_file() and p != pack and not p.name.startswith('epoch') and p.suffix != '.ipynb':
            z.write(p, p.relative_to(OUTPUT))
print(pack, round(pack.stat().st_size / 1e6, 1), 'MB')
for p in (RUN_DIR / 'weights').glob('epoch*.pt'):
    p.unlink()   # hemat ruang Output
print('Model terpilih:', CHOSEN, '|', status)